In [1]:
import pandas as pd

df_link_notna_text_na = pd.read_csv('filtered_data.csv')
print(df_link_notna_text_na.shape)
df_link_notna_text_na = df_link_notna_text_na[df_link_notna_text_na['link_text'].notna() & df_link_notna_text_na['text'].isna()]
df_link_notna_text_na.shape

(7864, 21)


(7864, 21)

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import concurrent.futures

def get_text_(id_link):
    id_, link = id_link
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 11_1_0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.146 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
    }
    session = requests.Session()
    session.headers = headers
    try:
        response = session.get(link, timeout=120)
        response.raise_for_status()
        bs = BeautifulSoup(response.text, 'html.parser')
        
        div = bs.find('div', id='content') 
        if not div:
            div = bs.find('div', class_='container')
        if div:
            text = div.get_text(separator="\n", strip=True)
            return [id_, link, text]
        else:
            print(f"not find text, link: {link}, status: {response.status_code}")
            return [id_, link, None]
    except requests.exceptions.RequestException as e:
        print(f"request error, link: {link}, error: {str(e)}")
        return [id_, link, None]
    except Exception as e:
        print(f"other error, link: {link}, type: {type(e)}, error: {str(e)}")
        return [id_, link, None]

def find_text(cur_df):
    links = list(zip(cur_df['id'], cur_df['link_text']))
    CONNECTIONS = 4
    out = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=CONNECTIONS) as executor:
        future_to_link = (executor.submit(get_text_, id_link) for id_link in links)
        for future in tqdm(concurrent.futures.as_completed(future_to_link), total=len(links)):
            try:
                data = future.result() 
            except Exception as exc:      
                data = str(type(exc))
            finally:
                out.append(data)
    return out

out = find_text(df_link_notna_text_na)
df_with_link_text = pd.DataFrame(out, columns=['id', 'link_text', 'text'])

  1%|          | 64/7864 [01:10<12:29:24,  5.76s/it]

not find text, link: http://baltachevsky--bkr.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=67024303&delo_id=1540006&new=0&text_number=1, status: 200


  1%|          | 68/7864 [01:22<6:41:17,  3.09s/it] 

not find text, link: http://meleuzovsky--bkr.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=15743993&delo_id=1540006&new=0&text_number=1, status: 200


  2%|▏         | 143/7864 [03:16<2:05:36,  1.02it/s]

...

 94%|█████████▎| 7366/7864 [2:33:44<33:40,  4.06s/it]

not find text, link: http://torzhoksky--twr.sudrf.ru/modules.php?name=sud_delo&name_op=case&_uid=1fbd26ba-d6d3-481f-96cc-499150342e7d&_deloId=1540006&_caseType=0&_new=0&_doc=1&srv_num=1&_hideJudge=0, status: 200


100%|█████████▉| 7845/7864 [2:41:22<00:43,  2.30s/it]

request error, link: http://sovetsky--mari.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=11367694&delo_id=1540006&new=0&text_number=1, error: 502 Server Error: Bad Gateway for url: https://sovetsky--mari.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=11367694&delo_id=1540006&new=0&text_number=1


100%|██████████| 7864/7864 [2:42:02<00:00,  1.24s/it]


In [3]:
print(df_with_link_text.shape)
df_with_link_text.head()

(7864, 3)


,id,link_text,text
0,102588,http://oblsud--kln.sudrf.ru/modules.php?name=s...,Решение по уголовному делу\nИнформация по делу...
1,24366,http://centr--cht.sudrf.ru/modules.php?name=su...,Решение по уголовному делу\nИнформация по делу...
2,24255,http://zabaykalsk--cht.sudrf.ru/modules.php?na...,Решение по уголовному делу\nИнформация по делу...
3,6258,http://pozharsky--prm.sudrf.ru/modules.php?nam...,Решение по уголовному делу
4,5018,http://leningradskay--krd.sudrf.ru/modules.php...,Решение по уголовному делу


In [4]:
df_with_link_text2 = df_with_link_text[df_with_link_text['link_text'].notna() & df_with_link_text['text'].isna()]
df_with_link_text2.shape

(129, 3)

In [5]:
out = find_text(df_with_link_text2)
df_with_link_text_new = pd.DataFrame(out, columns=['id', 'link_text', 'text'])

  2%|▏         | 2/129 [00:01<01:15,  1.68it/s]

not find text, link: http://torzhoksky--twr.sudrf.ru/modules.php?name=sud_delo&name_op=case&_uid=8e574f2c-e874-410d-b202-0e0f44d2fb6c&_deloId=1540006&_caseType=0&_new=0&_doc=1&srv_num=1&_hideJudge=0, status: 200


  4%|▍         | 5/129 [00:01<00:30,  4.07it/s]

not find text, link: http://gribanovsky--vrn.sudrf.ru/modules.php?name=sud_delo&name_op=case&_uid=bf8cf35d-f2a3-480b-b4ee-80d2a6adf005&_deloId=1540006&_caseType=0&_new=0&_doc=1&srv_num=1&_hideJudge=0, status: 200
not find text, link: http://torzhoksky--twr.sudrf.ru/modules.php?name=sud_delo&name_op=case&_uid=79ba9464-8136-420a-bc8f-b865ffe5623d&_deloId=1540006&_caseType=0&_new=0&_doc=1&srv_num=1&_hideJudge=0, status: 200


  5%|▍         | 6/129 [00:02<00:31,  3.85it/s]

...

not find text, link: http://torzhoksky--twr.sudrf.ru/modules.php?name=sud_delo&name_op=case&_uid=1fbd26ba-d6d3-481f-96cc-499150342e7d&_deloId=1540006&_caseType=0&_new=0&_doc=1&srv_num=1&_hideJudge=0, status: 200


 99%|█████████▉| 128/129 [01:45<00:00,  1.48it/s]

request error, link: http://todjinskiy--tva.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=53720529&delo_id=1540006&new=0&text_number=1, error: 404 Client Error: Not Found for url: https://todjinskiy--tva.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=53720529&delo_id=1540006&new=0&text_number=1


100%|██████████| 129/129 [01:47<00:00,  1.20it/s]

request error, link: http://orlovsky--ros.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=185882185&delo_id=1540006&new=0&text_number=1, error: 404 Client Error: Not Found for url: https://orlovsky--ros.sudrf.ru/modules.php?name=sud_delo&srv_num=1&name_op=doc&number=185882185&delo_id=1540006&new=0&text_number=1


In [6]:
print(df_with_link_text_new.shape)

df_with_link_text3 = df_with_link_text_new[df_with_link_text_new['link_text'].notna() & df_with_link_text_new['text'].isna()]
df_with_link_text3.shape

(129, 3)


(65, 3)

In [7]:
df_with_link_text_new[df_with_link_text_new['text'].notna()].head()

,id,link_text,text
0,2841,http://baltachevsky--bkr.sudrf.ru/modules.php?...,Решение по уголовному делу\nИнформация по делу...
3,3927,http://meleuzovsky--bkr.sudrf.ru/modules.php?n...,Решение по уголовному делу\nИнформация по делу...
25,61891,http://oblsud--mo.sudrf.ru/modules.php?name=su...,Решение по уголовному делу\nИнформация по делу...
26,65314,http://pervouralsky--svd.sudrf.ru/modules.php?...,Решение по уголовному делу\nИнформация по делу...
29,57880,http://pinegasud--arh.sudrf.ru/modules.php?nam...,Решение по уголовному делу\nИнформация по делу...


In [8]:
df_merged = df_with_link_text.merge(df_with_link_text_new, on="id", how="left", suffixes=("", "_new"))
df_merged["text"] = df_merged["text"].fillna(df_merged["text_new"])
df_with_link_text_final = df_merged[["id", "link_text", "text"]]
df_with_link_text_final.head()

,id,link_text,text
0,102588,http://oblsud--kln.sudrf.ru/modules.php?name=s...,Решение по уголовному делу\nИнформация по делу...
1,24366,http://centr--cht.sudrf.ru/modules.php?name=su...,Решение по уголовному делу\nИнформация по делу...
2,24255,http://zabaykalsk--cht.sudrf.ru/modules.php?na...,Решение по уголовному делу\nИнформация по делу...
3,6258,http://pozharsky--prm.sudrf.ru/modules.php?nam...,Решение по уголовному делу
4,5018,http://leningradskay--krd.sudrf.ru/modules.php...,Решение по уголовному делу


In [9]:
print(df_with_link_text_final.shape)
df_with_link_text4 = df_with_link_text_final[df_with_link_text_final['link_text'].notna() & df_with_link_text_final['text'].isna()]
df_with_link_text4.shape

(7864, 3)


(65, 3)

In [10]:
df_with_link_text_final.to_csv('df_with_link_text_final.csv', index=False)